# Chapter 21: Variational Quantum Linear Solver --- Drivers

The Variational Quantum Linear Solver (VQLS) tackles the quantum linear system problem --- find $|u\rangle=|x/\|x\|\rangle$ where $A\mathbf{x}=\mathbf{b}$ --- with a hybrid quantum-classical loop (Book Chapter 21). A parametric **ansatz** generates trial states $|u(\theta)\rangle$, $A$ is expanded as a linear combination of Pauli operators, and a quantum **cost function** scores each trial (lowest when $A|u(\theta)\rangle\propto|b\rangle$); a classical optimizer searches for the best $\theta^{*}$.

The reusable class lives in `Chapter21_VQLS_functions.py`. It implements the **global cost**
$$C_G(\theta)=1-\frac{|\langle b|A|u(\theta)\rangle|^2}{\langle u(\theta)|A^\dagger A|u(\theta)\rangle},$$
evaluated exactly from the statevector. The numerator is measured as $\langle u|M|u\rangle$ with the Hermitian operator $M=A^\dagger|b\rangle\langle b|A$, since $|\langle b|A|u\rangle|^2=\langle u|M|u\rangle$.

**Simulator vs. hardware.** Forming $M$ requires the projector $|b\rangle\langle b|$ (i.e. knowing $b$ classically), which is fine for these statevector demonstrations but is exactly what a real device cannot do at scale. On hardware, $\langle b|A|u\rangle$ must be estimated with the ancilla-based **Hadamard test** of Book §21.4. VQLS is a NISQ-era variational method: no proven quantum advantage, prone to barren plateaus at scale, competitive only on problems already easy classically. See `Chapter02_QuantumSoftware.ipynb` for installation.

## Setup and imports

In [1]:
import numpy as np
from scipy.sparse import diags
from Chapter21_VQLS_functions import VQLS

## Choose from $A\mathbf{x}=\mathbf{b}$ examples  *(Book §21.8)*

Each option fixes a matrix $A$ and right-hand side $b$. Example 5 (with `p`) is deliberately ill-conditioned; example 6 is an $N=8$ tridiagonal (Poisson-like) matrix --- a discretized differential operator, the structured regime VQLS is actually suited to. Select one via `example`.

In [2]:
example = 2
if example == 1:
    A = np.array([[1, 0], [0, 1]]);            b = np.array([1, 0])
elif example == 2:
    A = np.array([[2, -1], [-1, 2]]);          b = np.array([1, 1]) / np.sqrt(2)
elif example == 3:
    A = np.array([[1, 0, 0, -0.5], [0, 1, 0, 0],
                  [0, 0, 1, 0], [-0.5, 0, 0, 1]]); b = np.array([1, 0, 0, 0])
elif example == 4:
    A = np.array([[1.5, 0.5], [0.5, 1.5]]);    b = np.array([1, 0])
elif example == 5:
    p = 1
    A = np.array([[5 * (10**p), -1], [-1, 5]]); b = np.array([1, 0])
elif example == 6:
    N = 8
    values = [-np.ones(N - 1), 2 * np.ones(N), -np.ones(N - 1)]
    A = diags(values, [-1, 0, 1]).toarray()
    b = np.zeros(N); b[0] = 1

print("A:\n", A)
print("b:\n", b)

A:
 [[ 2 -1]
 [-1  2]]
b:
 [0.70710678 0.70710678]


## Solve with the global cost  *(Book §21.1--21.2)*

We build a `VQLS` solver for the chosen system, minimize $C_G(\theta)$ with COBYLA, and compare the recovered $|u(\theta^*)\rangle$ against the classical solution by fidelity. For the $2\times2$ and $4\times4$ systems `reps=6` suffices; the $N=8$ Poisson matrix uses a deeper ansatz.

In [3]:
reps = 8 if A.shape[0] >= 8 else 6
solver = VQLS(A, b, reps=reps, entanglement='full')

u, result = solver.solve(seed=1, maxiter=4000)

x_exact = solver.classical_solution()
fid = solver.fidelity(x_exact, u)

print(f"Final cost        : {result.fun:.2e}")
print(f"Exact (normalized): {np.round(x_exact, 6)}")
print(f"Quantum solution  : {np.round(u.real, 6)}")
print(f"Solution fidelity : {fid:.4f}")

Final cost        : 7.46e-11
Exact (normalized): [0.707107 0.707107]
Quantum solution  : [0.707105 0.707109]
Solution fidelity : 1.0000


### Shot count and accuracy

The exact statevector cost above converges cleanly. On real hardware (or a shot-based estimator) the cost is noisy, and shot count directly controls accuracy: too few shots and the optimizer is misled near the minimum. Book §21.8 shows the same $4\times4$ system converging to a correct solution with 50,000 shots and a poor one with 10,000. The statevector solver here is the noise-free reference those runs are measured against.

## Engineering example: the 1D Poisson system  *(Book §21.9)*

The tridiagonal Poisson (steady-state diffusion) stiffness matrix $\mathbf{K}\mathbf{d}=\mathbf{f}$ is the structured, sparse regime where VQLS is actually viable: its Pauli decomposition has only $L=N$ terms (vs. the dense worst case $4^n=N^2$), and it is only mildly ill-conditioned ($\kappa\approx32$ for $N=8$). We solve the $N=8$ (3-qubit) system with a unit source at node 0 and check fidelity against the classical solution.

Note the recovered solution may appear as $-\mathbf{u}^*$: VQLS fixes the state only up to a global phase, which does not affect fidelity.

In [4]:
N = 8
values = [-np.ones(N - 1), 2 * np.ones(N), -np.ones(N - 1)]
K = diags(values, [-1, 0, 1]).toarray()   # tridiagonal Poisson stiffness
f = np.zeros(N); f[0] = 1.0                # unit source at node 0

solver = VQLS(K, f, reps=8, entanglement='full')
u, result = solver.solve(seed=1, maxiter=5000)

x_exact = solver.classical_solution()
print(f"Final cost        : {result.fun:.2e}")
print(f"Exact (normalized): {np.round(x_exact, 4)}")
print(f"Quantum solution  : {np.round(u.real, 4)}")
print(f"Solution fidelity : {solver.fidelity(x_exact, u):.4f}")

Final cost        : 2.10e-06
Exact (normalized): [0.5601 0.4901 0.4201 0.3501 0.2801 0.21   0.14   0.07  ]
Quantum solution  : [-0.5624 -0.4911 -0.4198 -0.3488 -0.2782 -0.2081 -0.1384 -0.0691]
Solution fidelity : 1.0000
